# RAG with GroqCloud + Qdrant

A small, beginner-friendly RAG pipeline:

**PDF text → chunks → embeddings → Qdrant → retrieve context → Groq LLM → answer**

We use:
- **GroqCloud** for text generation through the OpenAI-compatible API.
- **Qdrant** via `qdrant-client` for vector search.
- **Sentence Transformers** locally for embeddings.
- **PyMuPDF** to read PDF files.

The Groq model follows the example in the request: `qwen/qwen3.8-27b`.

## 1. Install packages


### Follow SETUP.md

## 2. Set API keys and connect

Set `GROQ_API_KEY` in your environment before running, or enter it when prompted. For Qdrant, this example uses local storage.

In [1]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your GroqCloud API key: ")

from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

print("GroqCloud client ready.")

GroqCloud client ready.


## 3. Load the embedding model and create Qdrant storage

The LLM and embedding model do different jobs: Groq generates the answer, while Sentence Transformers converts text into vectors for search.

We use a local Qdrant local server instance for the simplest setup.

In [2]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

from pathlib import Path

BASE_DIR = Path.cwd()
QDRANT_PATH = (BASE_DIR / ".."/ ".."/ "qdrant_storage").resolve()
PDF_PATH = (BASE_DIR / ".."/ ".."/ "documents" / "sample_employee_handbook.pdf").resolve()

QDRANT_PATH.mkdir(parents=True, exist_ok=True)


embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vector_size = embedding_model.get_embedding_dimension()



qdrant = QdrantClient(path=str(QDRANT_PATH))
collection_name = "simple_rag"

if not qdrant.collection_exists(collection_name):
    qdrant.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE
        )
    )
    print(f"Created collection: {collection_name}")
else:
    print(f"Collection already exists: {collection_name}")

print("Embedding dimensions:", vector_size)


/Users/avikdasgupta/Documents/VSCODE/RAG_chatbot_YT/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1021 18:29:17.676000 27304 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8271.68it/s]


Created collection: simple_rag
Embedding dimensions: 384


## 4. Add a sample document

To keep the first run simple, we'll use a small text document. Later, replace it with text extracted from your own PDFs.

In [4]:
import pymupdf

def read_pdf(pdf_path):
    page_texts = []
    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()
            if text:
                page_texts.append({
                    "source": f"{pdf_path} (page {page_number})",
                    "text": text,
                })
    return page_texts

# Example:
pdf_documents = read_pdf(pdf_path=PDF_PATH)

## 5. Split documents into chunks

In [5]:
def split_into_chunks(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())

        if end >= len(text):
            break

        start = end - overlap

    return chunks

chunks = []
for doc in pdf_documents:
    for chunk in split_into_chunks(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["source"],
        })

print("Number of chunks:", len(chunks))
for chunk in chunks:
    print("\n", chunk)

Number of chunks: 46

 {'text': 'Page 1\nEMPLOYEE HANDBOOK\nSample Policies, Workplace Standards & Procedures\nOrganization\nExample Company (fictional)\nVersion\n1.0 — Sample template\nEffective date\nTo be completed by the organization\nPolicy owner\nHuman Resources\nImportant template notice\nThis handbook is a fictional sample intended for drafting, training, and document-search testing. It is not legal advice and\nis not ready for adoption without review. Employment requirements vary by jurisdiction, contract, workforce, and industry.', 'source': '/Users/avikdasgupta/Documents/VSCODE/RAG_chatbot_YT/documents/sample_employee_handbook.pdf (page 1)'}

 {'text': 'on without review. Employment requirements vary by jurisdiction, contract, workforce, and industry.\nThe organization should have qualified HR and legal reviewers adapt these policies, fill in local details, and ensure\nconsistency with applicable law and existing agreements.', 'source': '/Users/avikdasgupta/Documents/VSCODE/

## 6. Delete Existing chunks (to avoid duplication) and Upsert the chunks (points) into qdrant. 

In [6]:
from uuid import uuid4

texts = [chunk["text"] for chunk in chunks]
vectors = embedding_model.encode(texts, normalize_embeddings=True).tolist()
print(vectors[0])
points = []
for chunk, vector in zip(chunks, vectors):
    points.append(
        models.PointStruct(
            id=str(uuid4()),
            vector=vector,
            payload={
                "text": chunk["text"],
                "source": chunk["source"],
            },
        )
    )

# qdrant.delete(
#     collection_name=collection_name,
#     points_selector=models.FilterSelector(
#         filter=models.Filter(must=[])
#     ),
#     wait=True,
# )

qdrant.upsert(
    collection_name=collection_name,
    points=points,
    wait=True,
)

print(f"Saved {len(points)} chunks in Qdrant.")

[-0.050794828683137894, 0.06838496774435043, 0.011110673658549786, -0.04081372171640396, 0.06035695970058441, 0.043828874826431274, -0.006161061581224203, -0.03387748450040817, -0.024142378941178322, 0.005097686778753996, -0.0018965642666444182, 0.05098332464694977, -0.04390532150864601, 0.015306813642382622, 0.04225445166230202, -0.06262309104204178, 0.015710793435573578, -0.01280027162283659, 0.06173844262957573, -0.02410823665559292, 0.05322999879717827, -0.009451381862163544, -0.004004356451332569, -0.012030637823045254, -0.08787230402231216, -0.03946923092007637, 0.03718871250748634, 0.05220864713191986, -0.027715325355529785, -0.05480918660759926, -0.0011461596004664898, 0.012613496743142605, 0.0901392251253128, -0.062273841351270676, 0.11766035109758377, 0.05497918650507927, 0.01509856153279543, 0.025379600003361702, 0.012355233542621136, 0.04257725551724434, -0.04703697934746742, -0.06836501508951187, -0.04037213325500488, 0.03212442249059677, -0.016385875642299652, 0.001526254

## 7. Retrieve relevant chunks for a question

In [7]:
def retrieve(question, top_k=5):
    question_vector = embedding_model.encode(
        question,
        normalize_embeddings=True,
    ).tolist()

    results = qdrant.query_points(
        collection_name=collection_name,
        query=question_vector,
        limit=top_k,
        with_payload=True,
    ).points

    return [
        {
            "text": result.payload["text"],
            "source": result.payload["source"],
            "score": result.score,
        }
        for result in results
    ]

question = "What is the purpose of the employee handbook? what prohibited conduct related to Anti-Harassment and Bullying is mentioned in the handbook?"
for item in retrieve(question,10):
    print(item["source"], f"(score: {item['score']:.3f})")
    print(item["text"], "\n")

/Users/avikdasgupta/Documents/VSCODE/RAG_chatbot_YT/documents/sample_employee_handbook.pdf (page 2) (score: 0.650)
EXAMPLE COMPANY | EMPLOYEE HANDBOOK
Page 2
Contents
1. Purpose, Scope & Guiding Principles
2. Employment Practices & Equal Opportunity
3. Professional Conduct & Conflicts of Interest
4. Working Hours, Attendance & Remote Work
5. Leave, Holidays & Time Off
6. Compensation, Expenses & Benefits
7. Performance, Learning & Career Development
8. Workplace Health, Safety & Security
9. Anti-Harassment, Bullying & Reporting
10. Information Security, Privacy & Acceptable Use
11. Confidentiality, Records & 

/Users/avikdasgupta/Documents/VSCODE/RAG_chatbot_YT/documents/sample_employee_handbook.pdf (page 2) (score: 0.612)
handbook summarizes example workplace expectations and administrative procedures for employees of Example
Company. It is intended to help employees understand common processes, where to seek support, and how to raise
concerns. It applies to employees unless a written

## 8. Ask Groq to answer using the retrieved context

This uses the GroqCloud call pattern supplied for this tutorial. We retrieve context first, then pass the question and context to the model.

In [8]:
def ask_rag(question):
    retrieved_chunks = retrieve(question, 10)

    context = "\n\n".join(
        f"Source: {item['source']}\n{item['text']}"
        for item in retrieved_chunks
    )

    prompt = f'''
Answer the question using only the context below.
If the answer is not in the context, say "Not Enough Information".

- Do not make up an answer.
- Write the answer in a concise and clear manner and it should be readable and understandable. use md formatting for the answer.

Context:
{context}

Question:
{question}

# output format:
# Answer: <your answer here>
# Sources:[ <source1>, <source2>, ...]
'''

    response = client.responses.create(
        input=prompt,
        model="qwen/qwen3.8-27b",
        stream=True,
        max_output_tokens=300,
    )
    for event in response:
        if event.type == "response.output_text.delta":
            print(event.delta, end="")

ask_rag("What is the purpose of the employee handbook??")

# Answer: The employee handbook is intended to summarize workplace expectations and administrative procedures, help employees understand common processes, and guide them on where to seek support and how to raise concerns.
# Sources:[ /Users/avikdasgupta/Documents/VSCODE/RAG_chatbot_YT/documents/sample_employee_handbook.pdf (page 2) ]

## 9. Try another question

Try questions answered by the sample documents, and one that is not covered. Retrieval can return loosely related text even when the answer is absent, so inspect the retrieved chunks and don't assume RAG guarantees correctness.

In [9]:
ask_rag("What is the purpose of the employee handbook? what prohibited conduct related to Anti-Harassment and Bullying is mentioned in the handbook? ?")

# Answer:

**Purpose of the Employee Handbook**
The handbook summarizes example workplace expectations and administrative procedures for employees of Example Company. It is intended to help employees understand:
*   Common processes
*   Where to seek support
*   How to raise concerns

**Prohibited Conduct Related to Anti-Harassment and Bullying**
The handbook lists the following prohibited conduct:
*   Harassment
*   Sexual harassment
*   Bullying
*   Threats
*   Discrimination
*   Retaliation

# Sources:[ /Users/avikdasgupta/Documents/VSCODE/RAG_chatbot_YT/documents/sample_employee_handbook.pdf (page 2), /Users/avikdasgupta/Documents/VSCODE/RAG_chatbot_YT/documents/sample_employee_handbook.pdf (page 5) ]

## Recap

1. Split documents into chunks.
2. Embed each chunk.
3. Store the vectors and text in Qdrant.
4. Embed the question and retrieve relevant chunks.
5. Send the retrieved context to GroqCloud.
6. Stream the generated answer.


In [10]:
qdrant.close()